In [1]:
#Hook Colab up to Drive and set where results and figures get saved

import sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/DeepMinds')
sys.path.append(str(DATA_DIR))
FIG_DIR = DATA_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
LOG = DATA_DIR / 'results_log.csv'

Mounted at /content/drive


In [2]:
#Load the data + show the CNN's default settings
import torch
from project_utils import load_master, evaluate_model, log_result
from torch_models import fit_char_cnn, predict_proba, count_params, DEFAULT_CNN, DEVICE

df = load_master(DATA_DIR)
print('Running on', DEVICE)
print('Defaults:', DEFAULT_CNN)

Running on cuda
Defaults: {'max_len': 200, 'embed_dim': 32, 'filters': 64, 'kernel_sizes': (3, 5, 7), 'dropout': 0.3, 'hidden': 64, 'use_conv': True, 'strip_prefix': False, 'lr': 0.001, 'batch_size': 256, 'max_epochs': 20, 'patience': 3}


In [3]:
#Changing ONE setting per run, keep the rest at default, and loging every run
SWEEP = [
    ('kernel_sizes', (3,)), ('kernel_sizes', (3, 5, 7)),
    ('embed_dim', 16), ('embed_dim', 32), ('embed_dim', 64),
    ('filters', 32), ('filters', 64), ('filters', 128),
    ('max_len', 100), ('max_len', 200),
]

for setting, value in SWEEP:
    model, hist, cfg, seqs, _ = fit_char_cnn(df, {setting: value}, verbose=False)
    X_val, y_val = seqs['val']
    res = evaluate_model(f'Char-CNN {setting}={value}', 'val', y_val,
                         predict_proba(model, X_val, x_dtype=torch.long), show=False)
    log_result(res, {'family': 'CNN', 'experiment': 'sweep', 'swept': setting, 'features': 'raw URL',
                     'split_col': 'split', 'params': count_params(model),
                     'best_epoch': hist['best_epoch'], **cfg}, LOG)
    print(f'{setting}={value}:  F1 {res["F1"]:.4f}  recall {res["Recall"]:.4f}  '
          f'missed {res["False negatives"]}  epochs {len(hist["loss"])}  params {count_params(model):,}')

kernel_sizes=(3,):  F1 0.9977  recall 0.9954  missed 69  epochs 7  params 12,385
kernel_sizes=(3, 5, 7):  F1 0.9979  recall 0.9958  missed 63  epochs 8  params 45,281
embed_dim=16:  F1 0.9977  recall 0.9957  missed 65  epochs 8  params 28,945
embed_dim=32:  F1 0.9979  recall 0.9958  missed 63  epochs 8  params 45,281
embed_dim=64:  F1 0.9979  recall 0.9959  missed 62  epochs 8  params 77,953
filters=32:  F1 0.9977  recall 0.9956  missed 67  epochs 8  params 23,681
filters=64:  F1 0.9978  recall 0.9957  missed 65  epochs 8  params 45,281
filters=128:  F1 0.9977  recall 0.9956  missed 66  epochs 8  params 88,481
max_len=100:  F1 0.9978  recall 0.9959  missed 62  epochs 8  params 45,281
max_len=200:  F1 0.9978  recall 0.9957  missed 65  epochs 8  params 45,281


Observations: That changing one CNN setting at a time barely moved the results and every run missed between 62 and 69 of 15,142 phishing sites. Runs with identical default settings ranged from 63 to 65 missed, so differences of a few sites are just random variation between training runs. Even the smallest model (12,385 parameters) and the largest (88,481) performed about the same, and cutting URLs to 100 characters did not hurt. No setting came close to the URL-feature logistic regression (11 missed), so the character-level CNN does not beat hand-crafted URL features on this dataset.